# R3h version 89 report recovery
CPU only. Internet and existing KAGGLE_API_TOKEN, HF_TOKEN, GITHUB_TOKEN secrets. One Run All. Downloads selected cloud outputs within Kaggle; no local download and no training.

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

SOURCE_SHA='9c912a995810dca0cac3c74e15e38fd55b5459ec'
ROOT = Path("/kaggle/working/r3h-code-" + SOURCE_SHA[:12])
OUTPUT = Path("/kaggle/working/r3h-output")
ASSETS = Path("/kaggle/working/r3h-calibration-assets")


def run(*args, **kwargs):
    print("[R3h]", *map(str, args), flush=True)
    if str(args[0]) == "git":
        kwargs.setdefault("timeout", 45)
    return subprocess.run(list(map(str, args)), check=True, **kwargs)


# Use installed PyTorch; only add the existing storage dependency if absent.
if importlib.util.find_spec("yaml") is None:
    run(sys.executable, "-m", "pip", "install", "--quiet", "pyyaml")
if importlib.util.find_spec("huggingface_hub") is None:
    run(sys.executable, "-m", "pip", "install", "--quiet", "huggingface_hub>=0.23,<2")
ROOT.mkdir(parents=True, exist_ok=True)
run("git", "init", "--quiet", ROOT)
remote = subprocess.run(
    ["git", "-C", str(ROOT), "remote", "get-url", "origin"], capture_output=True, text=True
)
if remote.returncode:
    run("git", "-C", ROOT, "remote", "add", "origin", "https://github.com/ArcheLabs/mini-cells.git")
run("git", "-C", ROOT, "config", "remote.origin.promisor", "true")
run("git", "-C", ROOT, "config", "remote.origin.partialclonefilter", "blob:none")
run("git", "-C", ROOT, "sparse-checkout", "init", "--no-cone")
run(
    "git",
    "-C",
    ROOT,
    "sparse-checkout",
    "set",
    "--no-cone",
    *[
        "/scripts/run_r3h.py",
        "/pyproject.toml",
        "/src/",
        "/shared/python/",
        "/research/projects/cellular-language-models/engineering/kaggle-continuation/",
        "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/",
        "/research/projects/cellular-language-models/experiments/006-path-first-clm/",
        "/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/implementation/",
        "/research/projects/continual-learning/context/04-continual-learning/scripts/research/",
        "/research/projects/cellular-language-models/context/08-hybrid-clm/scripts/research/",
    ],
)
head = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "HEAD"], capture_output=True, text=True)
if head.returncode or head.stdout.strip() != SOURCE_SHA:
    run("git", "-C", ROOT, "fetch", "--depth=1", "--filter=blob:none", "origin", SOURCE_SHA)
    run("git", "-C", ROOT, "checkout", "--detach", "FETCH_HEAD")
actual = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
if actual != SOURCE_SHA:
    raise RuntimeError("Source identity mismatch")
env = dict(
    os.environ,
    HF_HUB_DISABLE_PROGRESS_BARS="1",
    HF_HUB_ETAG_TIMEOUT="10",
    HF_HUB_DOWNLOAD_TIMEOUT="30",
    TOKENIZERS_PARALLELISM="false",
    OMP_NUM_THREADS="1",
    MKL_NUM_THREADS="1",
)

# Recover only version 89's identity, result records, latest receipts and referenced objects.
import runpy, json
runpy.run_path(str(ROOT/'research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3h/run.py'),run_name='recovery_bootstrap')
from kaggle_secrets import UserSecretsClient
for label in ('KAGGLE_API_TOKEN','HF_TOKEN','GITHUB_TOKEN'):
    if not os.environ.get(label):
        os.environ[label]=UserSecretsClient().get_secret(label).strip()
if importlib.util.find_spec('kaggle') is None:
    run(sys.executable,'-m','pip','install','--quiet','git+https://github.com/Kaggle/kaggle-api.git@6894864ff4095cf98e9e36d3a694ff5fc9ba3818')
sys.path.insert(0,str(ROOT/'research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3b/implementation'))
from kaggle_posthoc.kaggle_source import KaggleOutputAPI,inventory
api=KaggleOutputAPI()
meta=api.metadata(89)
if meta.get('version_number') is not None and int(meta['version_number'])!=89:
    raise RuntimeError('Version mismatch')
files=[]
token=None
while True:
    page=api.page(89,token)
    files.extend(item['file_name'] for item in page['files'])
    token=page.get('next_page_token')
    if not token: break
prefix='r3h-output/'
selected=[f for f in files if f.startswith(prefix) and (f.endswith('/result.json') or f.endswith('/latest.json') or '/receipts/' in f or f in (prefix+'identity.json',prefix+'hf-reference.json'))]
if prefix+'identity.json' not in selected:raise RuntimeError('VERSION_89_R3H_OUTPUT_MISSING')
destination=Path('/kaggle/working/r3h-recovered-v89')
api.download(89,selected,destination)
OUTPUT=destination/'r3h-output'
objects=[]
for latest in (OUTPUT/'private-checkpoints/jobs').glob('*/latest.json'):
    row=json.loads(latest.read_text())
    relative=str((latest.parent/row['file']).relative_to(destination))
    if relative not in files:raise RuntimeError('LATEST_CHECKPOINT_NOT_RETAINED')
    objects.append(relative)
api.download(89,sorted(set(objects)),destination)
print('Recovered version 89; report-only; optimizer updates=0',flush=True)
run(sys.executable,ROOT/'scripts/run_r3h.py','--report-existing','--output',OUTPUT,cwd=ROOT,env=dict(os.environ))
print('Save Version Output:',destination,flush=True)
